# Demo 3 — Lo stesso clip in tre rappresentazioni

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

Il campione è un clip reale UrbanSound8K (air_conditioner), lo stesso primo clip del manifest didattico. La demo di rappresentazione usa soltanto questo campione; il protocollo completo resta nel notebook Jupyter del capitolo.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
y = al.load_clip(DEMO_ASSETS / "urban-demo.wav", sr=FS)
_, potenza = al.stft_power(y, FS, n_fft=1024, hop=512)
A = np.sqrt(potenza)
B = al.mel_filterbank(FS, n_fft=1024, n_mels=64, htk=False, norm="slaney")
mel = B @ A**2
logmel = al.log_mel(y, FS, n_mels=64, n_fft=1024, hop=512)
mfcc = al.mfcc_matrix(y, FS, n_mfcc=13, n_mels=64, n_fft=1024, hop=512)
print("Banco:", B.shape, "potenza:", A.shape, "log-mel:", logmel.shape, "MFCC:", mfcc.shape)
fig, ax = plt.subplots(figsize=(9, 3))
f = al.analysis.fft_frequencies(sr=FS, n_fft=1024)
for banda in B[::4]: ax.plot(f, banda)
ax.set(xlabel="Frequenza [Hz]", ylabel="Peso", title="Una banda ogni quattro: banco Slaney normalizzato per area")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for ax, S, asse_y, titolo in zip(axes,
    [al.analysis.power_to_db(A**2, ref=1.0), logmel, mfcc],
    ["linear", "mel", None], ["Potenza STFT in dB", "Log-mel: 64 bande", "MFCC: coefficienti 0–12"]):
    img = al.analysis.display.specshow(S, sr=FS, hop_length=512, x_axis="time",
                                  y_axis=asse_y, ax=ax)
    ax.set_title(titolo)
    fig.colorbar(img, ax=ax)
axes[-1].set_ylabel("Indice del coefficiente")
plt.tight_layout(); plt.show()
